# Autoresearch Experiment Runner & Analysis

This notebook is the **one-stop control panel** for the Gaudi autoresearch loop. It lets you:

1. **Configure** the connection to your Gaudi backend and local OpenAI-compatible API (via a `.env` file).
2. **Preflight-check** that everything is reachable before you burn compute.
3. **Run** the autonomous experiment loop (`harness/run_loop.py`).
4. **Analyze** the resulting `results.tsv`.

## What you need to set up

The loop has two moving parts that this notebook ties together:

| Piece | What it is | Where it's configured |
| --- | --- | --- |
| **The "brain"** | A local **OpenAI-compatible** chat endpoint (e.g. vLLM on Gaudi) that proposes edits to `train.py`. | `OPENAI_BASE_URL`, `OPENAI_API_KEY`, `OPENAI_MODEL` |
| **The "hands"** | The **Intel Gaudi 2 (HPU)** that actually runs each 5‑minute training experiment. | `AUTORESEARCH_RUNNER` (+ `AUTORESEARCH_NAMESPACE` / `AUTORESEARCH_POD` for k8s) |

### Two ways to run the "hands"

The scorer can execute `train.py` in either of two modes, set by `AUTORESEARCH_RUNNER`:

- **`local`** — `train.py` runs **on this machine**. Use this when the notebook runs
  *on the Gaudi node or inside the Gaudi pod* (e.g. the JupyterLab template, where the
  repo, this notebook, and the HPU are co-located). No `kubectl` needed.
- **`k8s`** — the scorer `kubectl cp`s `train.py` into a separate debug pod and runs it
  there. Use this when the notebook runs *outside* the cluster.
- *(unset)* — auto-detect: `k8s` if `kubectl` + a Running pod are found, else `local`.

### → To make this work, do this once:

```bash
cp .env.example .env      # then edit .env with your values
```

Set at minimum `OPENAI_BASE_URL` (must end in `/v1`), `OPENAI_API_KEY`, and `OPENAI_MODEL`
to point at **your** local model server, and pick `AUTORESEARCH_RUNNER`. For the
in-pod JupyterLab setup, `AUTORESEARCH_RUNNER=local` is what you want.

The cells below load that `.env`, verify the connection, run the loop, and plot the results.


## 1. Load configuration from `.env`

This reads `.env` (falling back to any values already in your shell environment) and pushes
them into `os.environ` so the harness picks them up. The API key is masked when printed.
If you see `MISSING`, edit your `.env` file and re-run this cell.


In [ ]:
import os
from pathlib import Path

# Project root = directory containing this notebook.
ROOT = Path.cwd()


def load_dotenv(path: Path) -> dict:
    """Minimal .env parser (no external dependency).

    Lines like KEY=VALUE are loaded into os.environ unless the key is already
    set in the real environment (real env wins, so you can override per-shell).
    """
    loaded = {}
    if not path.exists():
        return loaded
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key, val = key.strip(), val.strip().strip('"').strip("'")
        loaded[key] = val
        os.environ.setdefault(key, val)  # real env takes precedence
    return loaded


env_path = ROOT / ".env"
loaded = load_dotenv(env_path)

if not env_path.exists():
    print("⚠️  No .env file found. Copy the template and edit it:\n")
    print("      cp .env.example .env\n")
    print("    Falling back to any variables already in your shell environment.\n")
else:
    print(f"✓ Loaded {len(loaded)} setting(s) from {env_path}\n")

# Resolve the settings the rest of the notebook uses.
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL", "")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "")
RUNNER = os.environ.get("AUTORESEARCH_RUNNER", "").strip().lower()  # local | k8s | "" (auto)
NAMESPACE = os.environ.get("AUTORESEARCH_NAMESPACE", "autoresearch")
POD = os.environ.get("AUTORESEARCH_POD", "autoresearch-debug")
EXPERIMENT = os.environ.get("AUTORESEARCH_EXPERIMENT", "experiments/gpt_train")
ITERATIONS = int(os.environ.get("AUTORESEARCH_ITERATIONS", "5"))

# Auto-detect the runner the scorer will use, mirroring scorer.detect_runner():
# k8s only if kubectl exists AND the debug pod is Running, else local.
import shutil, subprocess  # noqa: E402

def _detect_runner() -> str:
    if RUNNER in ("local", "k8s"):
        return RUNNER
    if shutil.which("kubectl"):
        try:
            r = subprocess.run(
                ["kubectl", "get", "pod", POD, "-n", NAMESPACE,
                 "-o", "jsonpath={.status.phase}"],
                capture_output=True, text=True, timeout=30,
            )
            if r.returncode == 0 and r.stdout.strip() == "Running":
                return "k8s"
        except Exception:
            pass
    return "local"

EFFECTIVE_RUNNER = _detect_runner()


def _mask(secret: str) -> str:
    if not secret:
        return "MISSING"
    return secret[:4] + "…" + secret[-2:] if len(secret) > 6 else "set"


print("Brain (OpenAI-compatible API):")
print(f"  OPENAI_BASE_URL = {OPENAI_BASE_URL or 'MISSING'}")
print(f"  OPENAI_API_KEY  = {_mask(OPENAI_API_KEY)}")
print(f"  OPENAI_MODEL    = {OPENAI_MODEL or 'MISSING'}")
print("\nHands (where train.py runs):")
print(f"  runner = {EFFECTIVE_RUNNER}" + (f"  (AUTORESEARCH_RUNNER={RUNNER})" if RUNNER else "  (auto-detected)"))
if EFFECTIVE_RUNNER == "k8s":
    print(f"  namespace = {NAMESPACE}")
    print(f"  pod       = {POD}")
else:
    print("  → train.py runs directly on THIS machine (must be a Gaudi node/pod)")
print("\nExperiment loop:")
print(f"  experiment = {EXPERIMENT}")
print(f"  iterations = {ITERATIONS}")

missing = [n for n, v in [
    ("OPENAI_BASE_URL", OPENAI_BASE_URL),
    ("OPENAI_API_KEY", OPENAI_API_KEY),
    ("OPENAI_MODEL", OPENAI_MODEL),
] if not v]
if missing:
    print(f"\n⚠️  Still missing: {', '.join(missing)} — set these in .env before running the loop.")
else:
    print("\n✓ Brain config looks complete.")


## 2. Preflight checks

Before launching a multi-hour loop, verify the two dependencies are actually reachable:

- the **training backend** is usable — for `local` runner, this machine is a Gaudi
  node and `train.py` is present; for `k8s` runner, the debug pod is `Running`, and
- the **OpenAI-compatible API** answers and serves the model you configured.

Fix any ✗ below before moving on.


In [ ]:
import json
import urllib.error
import urllib.request

ok = True

# --- Check 1: the training backend is usable --------------------------------
if EFFECTIVE_RUNNER == "k8s":
    # Remote debug pod must be Running and reachable via kubectl.
    try:
        r = subprocess.run(
            ["kubectl", "get", "pod", POD, "-n", NAMESPACE,
             "-o", "jsonpath={.status.phase}"],
            capture_output=True, text=True, timeout=60,
        )
        phase = r.stdout.strip()
        if phase == "Running":
            print(f"✓ [k8s] Gaudi pod {NAMESPACE}/{POD} is Running")
        else:
            ok = False
            print(f"✗ [k8s] Gaudi pod {NAMESPACE}/{POD} not Running (phase={phase or r.stderr.strip()!r})")
            print("    start it with:  kubectl apply -f autoresearch-debug.yaml")
    except FileNotFoundError:
        ok = False
        print("✗ [k8s] kubectl not found on PATH — needed to drive the debug pod.")
    except subprocess.TimeoutExpired:
        ok = False
        print("✗ [k8s] kubectl get pod timed out — is your cluster reachable?")
else:
    # Local: train.py runs here, so this machine must BE a Gaudi node/pod.
    train_py = (ROOT / EXPERIMENT).resolve() / ".." / ".." / "train.py"
    train_py = train_py.resolve()
    if train_py.is_file():
        print(f"✓ [local] artifact present: {train_py}")
    else:
        ok = False
        print(f"✗ [local] train.py not found at {train_py}")
    # HPU visibility check (best-effort — hl-smi is the Gaudi 'nvidia-smi').
    if shutil.which("hl-smi"):
        try:
            r = subprocess.run(["hl-smi", "-L"], capture_output=True, text=True, timeout=30)
            n = r.stdout.count("Module ID") or r.stdout.lower().count("hl-")
            print(f"✓ [local] hl-smi sees the HPU(s){f' (~{n} listed)' if n else ''}")
        except Exception as e:
            print(f"⚠️  [local] hl-smi present but errored: {e}")
    else:
        print("⚠️  [local] hl-smi not found — make sure this really is a Gaudi node "
              "(train.py will fail to import habana_frameworks otherwise).")

# --- Check 2: OpenAI-compatible API answers and serves the model ------------
if not OPENAI_BASE_URL or not OPENAI_API_KEY or not OPENAI_MODEL:
    ok = False
    print("✗ Brain config incomplete — set OPENAI_* in .env (see cell above).")
else:
    try:
        req = urllib.request.Request(
            f"{OPENAI_BASE_URL.rstrip('/')}/models",
            headers={"Authorization": f"Bearer {OPENAI_API_KEY}"},
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        served = [m.get("id") for m in data.get("data", [])]
        print(f"✓ API reachable at {OPENAI_BASE_URL} ({len(served)} model(s) served)")
        if served and OPENAI_MODEL not in served:
            ok = False
            print(f"✗ OPENAI_MODEL={OPENAI_MODEL!r} not in served models: {served}")
        else:
            print(f"✓ Model {OPENAI_MODEL!r} is available")
    except urllib.error.HTTPError as e:
        # Some gateways don't expose /models; a 404 there isn't necessarily fatal.
        if e.code == 404:
            print(f"⚠️  {OPENAI_BASE_URL}/models returned 404 (endpoint may not list models); "
                  "the loop will still try /chat/completions.")
        else:
            ok = False
            print(f"✗ API error {e.code} from {OPENAI_BASE_URL}/models: {e.reason}")
    except (urllib.error.URLError, TimeoutError, ConnectionError) as e:
        ok = False
        print(f"✗ Could not reach API at {OPENAI_BASE_URL}: {e}")

print()
print("✅ Preflight passed — ready to run the loop." if ok
      else "❌ Preflight failed — fix the ✗ items above before running the loop.")


## 3. Run the autonomous experiment loop

This launches `harness/run_loop.py` against the configured experiment. Each iteration the
"brain" proposes one edit to `train.py`, the Gaudi pod runs a fixed **5‑minute** training
job to score it, and the harness keeps or discards the change — appending a row to the
experiment's `results.tsv`.

> ⏱️ **This is long-running.** With the default 5 iterations and a warm baseline, expect
> roughly `(iterations + 2) × ~6 min`. Output streams live below. Interrupt the kernel to
> stop early; progress is already saved in `results.tsv`.

Adjust `ITERATIONS` in `.env` (or override `iterations=` below) to control how long it runs.


In [ ]:
import sys


def run_loop(experiment: str = EXPERIMENT, iterations: int = ITERATIONS,
             skip_baseline: bool = False):
    """Stream harness/run_loop.py for the given experiment.

    Inherits the OPENAI_* env vars loaded from .env above. Set skip_baseline=True
    to resume from a prior run's best instead of re-establishing the baseline.
    """
    exp_path = (ROOT / experiment).resolve()
    if not (exp_path / "experiment.json").exists():
        raise FileNotFoundError(f"No experiment.json under {exp_path}")

    cmd = [
        sys.executable, str(ROOT / "harness" / "run_loop.py"),
        "--experiment", str(exp_path),
        "--iterations", str(iterations),
    ]
    if skip_baseline:
        cmd.append("--skip-baseline")

    print(f"$ {' '.join(cmd)}\n", flush=True)
    proc = subprocess.Popen(
        cmd, cwd=str(ROOT), env=os.environ.copy(),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        proc.terminate()
        print("\n[interrupted] loop stopped; results so far are saved in results.tsv")
        raise
    finally:
        rc = proc.wait()
    print(f"\n[run_loop exited with code {rc}]")
    return rc


# Launch the loop. Re-run with run_loop(iterations=20) for a longer session,
# or run_loop(skip_baseline=True) to resume from a previous best.
run_loop()


## 4. Analyze results

Load and visualize the experiment log. This prefers the experiment's own
`results.tsv` (written by the harness, e.g. `experiments/gpt_train/results.tsv`)
and falls back to the top-level `results.tsv`. Re-run these cells any time during
or after a loop to see the latest progress.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Prefer the experiment's own results.tsv (written by the harness), then fall
# back to the top-level results.tsv. Both share the same 5 columns; only the
# first column header differs (hash vs commit), which we don't reference by name.
candidates = [
    ROOT / EXPERIMENT / "results.tsv",
    ROOT / "results.tsv",
]
results_path = next((p for p in candidates if p.exists()), None)
if results_path is None:
    raise FileNotFoundError(
        "No results.tsv found. Run the experiment loop above first, or check:\n  "
        + "\n  ".join(str(p) for p in candidates)
    )
print(f"Loading results from: {results_path}\n")

# Load the TSV (tab-separated, 5 columns: <hash/commit>, val_bpb, memory_gb, status, description)
df = pd.read_csv(results_path, sep="\t")
df["val_bpb"] = pd.to_numeric(df["val_bpb"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)


In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    bpb = row["val_bpb"]
    desc = row["description"]
    print(f"  #{i:3d}  bpb={bpb:.6f}  mem={row['memory_gb']:.1f}GB  {desc}")

## Val BPB Over Time

Track how the best (kept) val_bpb evolves as experiments progress. The running minimum shows the "frontier" -- the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_bpb = valid.loc[0, "val_bpb"]

# Only plot points at or below baseline (the interesting region)
below = valid[valid["val_bpb"] <= baseline_bpb + 0.0005]

# Plot discarded as faint background dots
disc = below[below["status"] == "DISCARD"]
ax.scatter(disc.index, disc["val_bpb"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_v = below[below["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["val_bpb"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running minimum step line
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_bpb = valid.loc[kept_mask, "val_bpb"]
running_min = kept_bpb.cummin()
ax.step(kept_idx, running_min, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment with its description
for idx, bpb in zip(kept_idx, kept_bpb):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."

    ax.annotate(desc, (idx, bpb),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Validation BPB (lower is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

# Y-axis: from just below best to just above baseline
best_bpb = kept_bpb.min()
margin = (baseline_bpb - best_bpb) * 0.15
ax.set_ylim(best_bpb - margin, baseline_bpb + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats
kept = df[df["status"] == "KEEP"].copy()
baseline_bpb = df.iloc[0]["val_bpb"]
best_bpb = kept["val_bpb"].min()
best_row = kept.loc[kept["val_bpb"].idxmin()]

print(f"Baseline val_bpb:  {baseline_bpb:.6f}")
print(f"Best val_bpb:      {best_bpb:.6f}")
print(f"Total improvement: {baseline_bpb - best_bpb:.6f} ({(baseline_bpb - best_bpb) / baseline_bpb * 100:.2f}%)")
print(f"Best experiment:   {best_row['description']}")
print()

# How many experiments to find each improvement
print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: bpb={row['val_bpb']:.6f}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment's bpb
# (since experiments are cumulative -- each one builds on the last kept state)
kept = df[df["status"] == "KEEP"].copy()
kept["prev_bpb"] = kept["val_bpb"].shift(1)
kept["delta"] = kept["prev_bpb"] - kept["val_bpb"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'BPB':>10}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_bpb']:.6f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  TOTAL improvement over baseline")